In [1]:
import os
import glob
import json
import ast
import numpy as np
import pandas as pd
import torch
import timm
import sys

In [2]:
sys.path.append(os.path.abspath('../'))
from birdclef_utils.constants import NUM_CLASSES

In [3]:
def parse_label_list(value):
    if value is None:
        return []
    s = str(value).strip()
    if s in ('', '[]', 'nan', 'NaN', 'None'):
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, SyntaxError):
        pass
    return [c.strip() for c in s.replace(';', ' ').split() if c.strip()]

In [4]:
def build_multihot(codes, label2idx, num_classes=NUM_CLASSES):
    label = torch.zeros(num_classes, dtype=torch.float32)
    for code in codes:
        idx = label2idx.get(code)
        if idx is not None:
            label[idx] = 1.0
    return label

In [15]:
# --- Configuration ---
csv_path='../dataset/train.csv'
csv_path_soundcape='../dataset/train_soundscapes_labels.csv'
label2idx_path='../dataset/label2idx.json'

npy_folder='../dataset/epi_spectrogram/spec_focal/'
npy_folder1='../dataset/epi_spectrogram/spec_soundscape/'

save_path='../dataset/training_data_embeddings.npz'
batch_size=32  # Reduce to 16 or 8 if you run out of GPU memory

In [8]:
with open(label2idx_path, 'r') as f:
    label2idx = json.load(f)
num_classes = len(label2idx)

df = pd.read_csv(csv_path)
# Setting the filename as the index makes looking up rows O(1) instead of O(N)
df.set_index('filename', inplace=True)

print(f"Loaded {len(df)} rows from train.csv")
print(f"Number of target classes: {num_classes}")

Loaded 35549 rows from train.csv
Number of target classes: 234


In [9]:
 # Initialize the Pre-trained EfficientNetV2-S Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# num_classes=0 removes the final classification head to output raw embeddings
model = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=0)
model = model.to(device)

# Crucial: set to evaluation mode to disable dropout and freeze batch normalization
model.eval() 

Using device: cpu


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

C:\Users\Admin\.conda\envs\ml\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--timm--tf_efficientnetv2_s.in21k_ft_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


EfficientNet(
  (conv_stem): Conv2dSame(3, 24, kernel_size=(3, 3), stride=(2, 2), bias=False)
  (bn1): BatchNormAct2d(
    24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): ConvBnAct(
        (conv): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNormAct2d(
          24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (drop_path): Identity()
      )
      (1): ConvBnAct(
        (conv): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNormAct2d(
          24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (drop_path):

## Focal 

In [10]:
# Gather all .npy files
npy_files = glob.glob(os.path.join(npy_folder, '*.npy'))

print(f"Found {len(npy_files)} files")

all_embeddings = []
all_labels = []

print(f"Starting extraction...")

Found 35549 files. Starting extraction...
Starting extraction...


In [11]:
# Disable gradient tracking to save massive amounts of RAM
with torch.no_grad():
    # Process in batches
    for i in range(0, len(npy_files), batch_size):
        batch_files = npy_files[i:i+batch_size]
        batch_tensors = []
        batch_y = []

        for npy_file in batch_files:
            # Step A: Reconstruct the CSV filename from the .npy filename
            # Example: '22930__iNat317238.npy' -> '22930/iNat317238.ogg'
            base_name = os.path.basename(npy_file)
            ogg_filename = base_name.replace('.npy', '').replace('__', '/') + '.ogg'
            
            # Verify the file exists in our CSV
            if ogg_filename not in df.index:
                print(f"Warning: {ogg_filename} not found in CSV. Skipping.")
                continue 
            
            # Step B: Get labels and build Multi-Hot Vector (Y)
            row = df.loc[ogg_filename]
            
            # Combine primary and secondary labels
            primary = row['primary_label']
            secondary = row.get('secondary_labels', '[]')
            codes = [primary] + parse_label_list(secondary)
            
            # Create the target vector
            y_vec = build_multihot(codes, label2idx, num_classes=num_classes).numpy()
            batch_y.append(y_vec)
            
            # Step C: Load Spectrogram for the CNN (X)
            spec = np.load(npy_file)
            
            # Pre-trained CNNs expect 3 channels (RGB). 
            # We duplicate your 1-channel spectrogram 3 times -> (3, 128, 313)
            tensor_spec = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
            batch_tensors.append(tensor_spec)
            
        # If the entire batch was skipped (not in CSV), move to the next
        if not batch_tensors:
            continue

        # Stack into a single batch tensor -> (Batch_Size, 3, 128, 313)
        X_batch = torch.stack(batch_tensors).to(device)
        
        # Step D: Forward pass to extract embeddings
        embeddings = model(X_batch)
        
        all_embeddings.append(embeddings.cpu().numpy())
        all_labels.extend(batch_y)
        
        print(f"Processed {min(i+batch_size, len(npy_files))}/{len(npy_files)} files...")

Processed 32/35549 files...
Processed 64/35549 files...
Processed 96/35549 files...
Processed 128/35549 files...
Processed 160/35549 files...
Processed 192/35549 files...
Processed 224/35549 files...
Processed 256/35549 files...
Processed 288/35549 files...
Processed 320/35549 files...
Processed 352/35549 files...
Processed 384/35549 files...
Processed 416/35549 files...
Processed 448/35549 files...
Processed 480/35549 files...
Processed 512/35549 files...
Processed 544/35549 files...
Processed 576/35549 files...
Processed 608/35549 files...
Processed 640/35549 files...
Processed 672/35549 files...
Processed 704/35549 files...
Processed 736/35549 files...
Processed 768/35549 files...
Processed 800/35549 files...
Processed 832/35549 files...
Processed 864/35549 files...
Processed 896/35549 files...
Processed 928/35549 files...
Processed 960/35549 files...
Processed 992/35549 files...
Processed 1024/35549 files...
Processed 1056/35549 files...
Processed 1088/35549 files...
Processed 1120

## Soundscapes

In [16]:
# --- Load and Setup the Soundscape CSV ---
df = pd.read_csv(csv_path_soundcape)

# Create a unique lookup key combining filename and end time
# Example: 'BC2026_Train_0001_S08_20250606_030007.ogg_00:00:05'
df['lookup_key'] = df['filename'] + "_" + df['end']

# Set this new key as the index for O(1) instant lookups
df.set_index('lookup_key', inplace=True)

print(f"Loaded {len(df)} soundscape chunks into memory.")

Loaded 1478 soundscape chunks into memory.


In [17]:
npy_files = glob.glob(os.path.join(npy_folder1, '*.npy'))

print(f"Found {len(npy_files)} files")

print(f"Starting extraction...")

Found 739 files
Starting extraction...


In [18]:
with torch.no_grad():
    # Process in batches
    for i in range(0, len(npy_files), batch_size):
        batch_files = npy_files[i:i+batch_size]
        batch_tensors = []
        batch_y = []

        for npy_file in batch_files:
            # Step A: Reconstruct the CSV lookup key from the .npy filename
            # Example: 'BC2026_Train_0001_S08_20250606_030007_5.npy'
            base_name = os.path.basename(npy_file).replace('.npy', '')
            
            # Split off the last underscore to separate filename and seconds
            # parts[0] -> 'BC2026_Train_0001_S08_20250606_030007'
            # parts[1] -> '5'
            parts = base_name.rsplit('_', 1)
            ogg_filename = parts[0] + '.ogg'
            end_sec = int(parts[1])
            
            # Format the integer seconds into HH:MM:SS string
            hours = end_sec // 3600
            minutes = (end_sec % 3600) // 60
            seconds = end_sec % 60
            end_time_str = f"{hours:02d}:{minutes:02d}:{seconds:02d}"
            
            # Reconstruct the precise lookup key
            lookup_key = f"{ogg_filename}_{end_time_str}"
            
            # Verify the specific 5-second chunk exists in our CSV
            if lookup_key not in df.index:
                print(f"Warning: {lookup_key} not found in CSV. Skipping.")
                continue 
            
            # Step B: Get labels and build Multi-Hot Vector (Y)
            row = df.loc[lookup_key]
            
            # If there are duplicates in CSV for some reason, take the first one
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            
            # Soundscapes combine all labels into 'primary_label' separated by ';'
            primary = row['primary_label']
            codes = parse_label_list(primary)
            
            # Create the target vector
            y_vec = build_multihot(codes, label2idx, num_classes=num_classes).numpy()
            batch_y.append(y_vec)
            
            # Step C: Load Spectrogram for the CNN (X)
            spec = np.load(npy_file)
            
            # Pre-trained CNNs expect 3 channels (RGB). 
            # Duplicate 1-channel spectrogram 3 times -> (3, 128, 313)
            tensor_spec = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
            batch_tensors.append(tensor_spec)
            
        # If the entire batch was skipped (not in CSV), move to the next
        if not batch_tensors:
            continue

        # Stack into a single batch tensor -> (Batch_Size, 3, 128, 313)
        X_batch = torch.stack(batch_tensors).to(device)
        
        # Step D: Forward pass to extract embeddings
        embeddings = model(X_batch)
        
        all_embeddings.append(embeddings.cpu().numpy())
        all_labels.extend(batch_y)
        
        print(f"Processed {min(i+batch_size, len(npy_files))}/{len(npy_files)} files...")

Processed 32/739 files...
Processed 64/739 files...
Processed 96/739 files...
Processed 128/739 files...
Processed 160/739 files...
Processed 192/739 files...
Processed 224/739 files...
Processed 256/739 files...
Processed 288/739 files...
Processed 320/739 files...
Processed 352/739 files...
Processed 384/739 files...
Processed 416/739 files...
Processed 448/739 files...
Processed 480/739 files...
Processed 512/739 files...
Processed 544/739 files...
Processed 576/739 files...
Processed 608/739 files...
Processed 640/739 files...
Processed 672/739 files...
Processed 704/739 files...
Processed 736/739 files...
Processed 739/739 files...


## Saving

In [19]:
X_train = np.vstack(all_embeddings)
Y_train = np.array(all_labels)

print(f"\nExtraction complete!")
print(f"X (Features) shape: {X_train.shape}  -> Should be (N_samples, 1280)")
print(f"Y (Labels) shape:   {Y_train.shape}  -> Should be (N_samples, {num_classes})")

# Save to a compressed .npz file (Highly efficient)
np.savez_compressed(save_path, X=X_train, Y=Y_train)
print(f"Saved successfully to: {save_path}")


Extraction complete!
X (Features) shape: (36288, 1280)  -> Should be (N_samples, 1280)
Y (Labels) shape:   (36288, 234)  -> Should be (N_samples, 234)
Saved successfully to: ../dataset/training_data_embeddings.npz
